
# A2a — Cheap Temporal-Attention Probes (No Retrain)

**Status:** designed July 2026, following A3's negative result on the Koopman
lift's own geometry as an explanation for A1's Claim 2 pattern (Section 8).
Temporal attention is the last major untested architectural component
(channel attention: 4/4 independent nulls, Experiments 9/22/27/33; Koopman
lift geometry: ruled out as mechanism, A3).

**Question.** Does temporal attention's use of sequential/positional
structure track the same selective pattern A1 found (lift helps on Lorenz/
Rossler/SprottB/Harmonic, hurts on Burgers ν=1.0)?

**Three probes, all inference-time, no retraining:**
1. Patch-order shuffling — destroys sequence order, preserves per-patch content.
2. Context-length truncation curves — dose-response of advantage vs. available context.
3. Attention-map inspection — descriptive, using the paper's own Toeplitz/block/
   selector/hybrid vocabulary (Section 5.5 of the Panda paper).

**Checkpoints used:** both `baseline_100k` (lift present) and `ablation_100k`
(lift removed), per the confirmed decision to include the ablation arm as a
bonus comparison — this also tells us whether temporal attention's behaviour
itself depends on the lift being present.

**Classes:** the same five used in A3 (Lorenz `gate_3ch`, Rossler, SprottB,
Burgers ν=1.0, Harmonic) — kept identical to A1/A3 rather than introducing a
new confound. Burgers uses the full 16-channel PCA representation from
Section 7's `load_burgers_nu1` (NOT A3's restricted top-3 channels — A2a is
a forecasting-MAE experiment like A1, not an eDMD-geometry experiment like
A3, so it should match A1's own OOD protocol for comparability).

**n_windows = 20** throughout (confirmed), matching the A1/A3 confirmatory
standard rather than the older n=8 exploratory standard.

**Chronos is deliberately not loaded in this notebook.** All three probes
compare Panda against itself (vanilla vs. shuffled, full vs. truncated
context, baseline vs. ablation checkpoint) — none require a Panda-vs-
Chronos advantage number. This also sidesteps the transformers version
conflict entirely rather than requiring two-environment isolation within
this notebook (see Cell 1's note for the failure this avoids).

---

## Pre-registered decision criterion (fixed BEFORE running — do not edit after seeing results)

**Escalate to A2b (temporal-attention ablation retrain) if:**
shuffle-induced MAE degradation (`MAE_shuffled - MAE_vanilla`, Panda only,
`baseline_100k` checkpoint) on the three chaotic-ODE classes (Lorenz,
Rossler, SprottB) is significantly larger (paired Wilcoxon, n=20 windows,
one-sided, α=0.05) than on Burgers ν=1.0, in a **majority of tested
horizons** (i.e. ≥2 of 3 horizons in {96, 192, 336}).

**Do not escalate if:** degradation is uniform across classes, absent
everywhere, or the ordering is inconsistent/reversed across horizons.

This threshold is deliberately about the *shuffle probe only* — probes 2
and 3 are supporting/exploratory evidence, not part of the pre-registered
gate, consistent with this project's stated policy of not letting
descriptive/dose-response results silently become the decision criterion
after the fact.

**Caveat carried over from A1/A3:** single trajectory per system throughout.
Window-overlap in the paired Wilcoxon tests has not been corrected for
(the project-wide pending integrity item) — p-values here should be read
with that in mind, same as everywhere else in this log.



## Part 0 — Environment Setup

Following the established two-environment-isolation pattern: Panda needs
`transformers==4.40.2`; installing Chronos afterward will silently upgrade
it. Force-reinstall 4.40.2 after both are installed. `peft` must be
uninstalled (known conflict). **Restart the kernel after running Cell 1**
before proceeding — this fixes a numpy binary conflict seen throughout this
project's Kaggle sessions.


In [ ]:

# Cell 1 — install / pin dependencies. RESTART KERNEL AFTER THIS CELL.
# NOTE (fixed after a real failure, round 2): the original --no-deps flag
# was there to stop chronos-forecasting from re-upgrading transformers.
# Chronos is no longer installed in this notebook at all (previous fix),
# so --no-deps now serves no purpose and actively causes a NEW failure:
# it blocks pip from installing transformers' own required tokenizers
# version (>=0.19,<0.20), leaving Kaggle's base-image tokenizers (0.22.2)
# in place, which is incompatible with 4.40.2. Fix: drop --no-deps, pin
# both packages together so pip resolves them as a matched pair.
!pip install -q "transformers==4.40.2" "tokenizers>=0.19,<0.20" --force-reinstall
!pip uninstall -y -q peft
!pip install -q einops
print("Done. RESTART THE KERNEL NOW, then continue from Cell 2.")
print("Verify the pin held: run `!pip show transformers tokenizers` "
      "in a scratch cell before Cell 4 -- transformers should read 4.40.2, "
      "tokenizers should read 0.19.x.")


In [ ]:

# Cell 2 — imports (run after kernel restart)
import os, glob, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import wilcoxon
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.manual_seed(99)  # matches the project-wide seed convention
np.random.seed(99)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")



## Part 1 — Load Both Checkpoints (Checkpoint Locator)

Walks the dataset tree rather than assuming a fixed path, per the project's
established pattern (checkpoints from Kaggle runs land in unpredictable
`/kaggle/input/.../checkpoint-100000/` locations depending on dataset
version).


In [ ]:

# Cell 3 — checkpoint locator
def find_checkpoint(root, name_hint, step_hint="100000"):
    \"\"\"Walk root looking for a directory containing config.json whose
    path also contains name_hint and step_hint. Returns the first match,
    printing all candidates found so a human can sanity-check the choice.\"\"\"
    candidates = []
    for dirpath, dirnames, filenames in os.walk(root):
        if "config.json" in filenames:
            if name_hint.lower() in dirpath.lower() and step_hint in dirpath:
                candidates.append(dirpath)
    print(f"[{name_hint}] candidates found:")
    for c in candidates:
        print("  ", c)
    if not candidates:
        raise FileNotFoundError(f"No checkpoint found under {root} matching "
                                 f"name_hint={name_hint!r}, step_hint={step_hint!r}")
    return candidates[0]

KAGGLE_INPUT_ROOT = "/kaggle/input"  # adjust if running outside Kaggle

BASELINE_CKPT = find_checkpoint(KAGGLE_INPUT_ROOT, "baseline", "100000")
ABLATION_CKPT = find_checkpoint(KAGGLE_INPUT_ROOT, "ablation", "100000")

# Sanity check: confirm use_dynamics_embedding matches the expected arm for
# each checkpoint BEFORE running anything else, per the campaign's own
# stated practice (Section 7, 100k Evaluation) after the earlier
# baseline-vs-ablation identity confusion in that campaign.
for name, ckpt in [("BASELINE", BASELINE_CKPT), ("ABLATION", ABLATION_CKPT)]:
    with open(os.path.join(ckpt, "config.json")) as f:
        cfg = json.load(f)
    flag = cfg.get("use_dynamics_embedding", "MISSING")
    print(f"{name}: use_dynamics_embedding = {flag}  (expected: "
          f"{'True' if name=='BASELINE' else 'False'})")
    expected = True if name == "BASELINE" else False
    assert flag == expected, (
        f"STOP: {name} checkpoint's use_dynamics_embedding={flag} does not "
        f"match expectation. Do not proceed until this is resolved — this "
        f"is exactly the confusion that occurred in the original A1 100k "
        f"campaign (Section 7, Correction 1)."
    )


In [ ]:

# Cell 4 — load both Panda checkpoints using Panda's OWN classes.
# NOTE (fixed after a real failure, round 3): AutoModel(trust_remote_code=
# True) was never going to work. The checkpoint's config.json declares
# model_type="patchtst", so AutoModel resolves it straight to the STOCK
# transformers.models.patchtst implementation bundled in the library --
# which has no rmsnorm branch -- and never looks at any custom repo code
# at all. The actual Panda architecture lives in the cloned panda repo
# (panda.patchtst.patchtst.PatchTSTForPrediction), confirmed directly from
# the user's own training script. Loading now mirrors that script's own
# resume pattern: build the architecture fresh from the training-time
# config, then load weights via a strict state_dict load.

import sys, os, json

# Adjust this path if the panda repo isn't already present at this location
# in this Kaggle session (it must be cloned/attached the same way it was
# for the original training notebook -- this notebook does not clone it
# for you, since the exact source (git URL vs. Kaggle dataset) isn't known
# here).
sys.path.insert(0, '/kaggle/working/panda')

from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

# Verify transformers/tokenizers are still a sane pair -- panda's classes
# extend transformers.models.patchtst internals (PatchTSTConfig is
# imported from transformers directly in the training script), so a wildly
# mismatched transformers version could still break field compatibility
# even though we're bypassing AutoModel.
import subprocess
def get_pkg_version(pkg):
    out = subprocess.check_output(["pip", "show", pkg]).decode()
    return [l for l in out.splitlines() if l.startswith("Version:")][0].split()[-1]
print(f"transformers: {get_pkg_version('transformers')}, "
      f"tokenizers: {get_pkg_version('tokenizers')}")

# Base config, verbatim from the training script (21M model, matches the
# GilpinLab/panda checkpoint). use_dynamics_embedding is set per-arm below.
BASE_MODEL_CONFIG = dict(
    mode='predict', context_length=512, prediction_length=128,
    patch_length=16, patch_stride=16, num_hidden_layers=8, d_model=512,
    num_attention_heads=8, channel_attention=True, ffn_dim=512,
    norm_type='rmsnorm', norm_eps=1e-5, attention_dropout=0.0,
    positional_dropout=0.0, path_dropout=0.0, ff_dropout=0.0, bias=True,
    activation_function='gelu', pre_norm=True, use_cls_token=False,
    init_std=0.02, scaling='std', pooling_type='max', head_dropout=0.0,
    channel_rope=False, max_wavelength=500, rope_percent=0.75, loss='mse',
    distribution_output=None, num_poly_feats=120, poly_degrees=2,
    rff_trainable=False, rff_scale=1.0, num_rff=256, do_mask_input=None,
    mask_type='random', random_mask_ratio=0.5,
    channel_consistent_masking=False, mask_value=0,
    num_forecast_mask_patches=3, unmasked_channel_indices=None,
    num_parallel_samples=100,
)

def load_panda_checkpoint(ckpt_dir, use_dynamics_embedding, arm_label):
    cfg = dict(BASE_MODEL_CONFIG)
    cfg['use_dynamics_embedding'] = use_dynamics_embedding

    model = load_patchtst_model(
        mode='predict', model_config=cfg,
        pretrained_encoder_path=None, pretained_checkpoint=None,
    )

    info_path = os.path.join(ckpt_dir, 'training_info.json')
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = json.load(f)
        assert info['use_dynamics_embedding'] == use_dynamics_embedding, (
            f"ARM MISMATCH loading {arm_label}: checkpoint "
            f"use_dynamics_embedding={info['use_dynamics_embedding']} but "
            f"expected {use_dynamics_embedding}. Wrong checkpoint directory "
            f"-- stopping before use."
        )
        print(f"[{arm_label}] identity verified: run_name={info.get('run_name')}, "
              f"use_dynamics_embedding={info['use_dynamics_embedding']}")
    else:
        print(f"[{arm_label}] WARNING: training_info.json missing -- arm "
              f"identity NOT auto-verified. Confirm manually before trusting "
              f"any result from this checkpoint.")

    st_path = os.path.join(ckpt_dir, 'model.safetensors')
    bin_path = os.path.join(ckpt_dir, 'pytorch_model.bin')
    if os.path.exists(st_path):
        from safetensors.torch import load_file as load_sf
        state = load_sf(st_path)
    elif os.path.exists(bin_path):
        state = torch.load(bin_path, map_location='cpu')
    else:
        raise FileNotFoundError(f'No weights found in {ckpt_dir}')

    model.load_state_dict(state, strict=True)
    model = model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[{arm_label}] loaded, {n_params:,} parameters")
    return model

panda_baseline = load_panda_checkpoint(
    BASELINE_CKPT, use_dynamics_embedding=True, arm_label="baseline_100k")
panda_ablation = load_panda_checkpoint(
    ABLATION_CKPT, use_dynamics_embedding=False, arm_label="ablation_100k")

print("Both Panda checkpoints loaded via native classes.")


In [ ]:

# Cell 4c — wrap both loaded models in PatchTSTPipeline (confirmed
# constructor: takes mode + an already-built PatchTSTForPrediction model,
# does not load from a checkpoint path itself -- so this reuses Cell 4's
# arm-verified model objects directly rather than loading a second time).
from panda.patchtst.pipeline import PatchTSTPipeline

pipe_baseline = PatchTSTPipeline(mode="predict", model=panda_baseline)
pipe_ablation = PatchTSTPipeline(mode="predict", model=panda_ablation)

print("Both pipelines constructed.")



## Part 2 — MANDATORY: Source Inspection Before Proceeding

**Do not skip this section or assume the module structure below.** Every
prior confound in this project (A3's keyword-search false positive on the
lift module, the cross-channel patch-pairing bug, the `@torch.no_grad()`
wrapper on `predict()`) was caught by inspecting source directly rather
than assuming an API. Run Cell 5 and read its output before Cell 6, which
now builds directly on what Cell 5 confirms (8 encoder layers, each with
its own `temporal_self_attn`/`channel_self_attn`, both `PatchTSTRopeAttention`)
rather than asking you to guess a single module path.

Note on scope: an earlier version of this section also asked for a
position-embedding module path and a patchifier module path, intended for
Probe 1's shuffle-and-reindex step. Both turned out to be unnecessary once
`PatchTSTRopeAttention.forward`'s source was inspected: RoPE position is
computed from sequence length at attention time
(`self.get_seq_pos(src_len, ...)`), not read from a separately stored
position tensor. Shuffling the raw `(T, C)` array before it reaches the
patchifier (Cell 8) already achieves the reindexing Probe 1 needs, with no
separate module access required.


In [ ]:

# Cell 5 — source inspection (READ THE OUTPUT before continuing)
import inspect

print("=" * 70)
print("Top-level module tree (baseline checkpoint):")
print("=" * 70)
for name, module in panda_baseline.named_modules():
    depth = name.count(".")
    if depth <= 2 and name:
        print("  " * depth + name, "->", type(module).__name__)

print()
print("=" * 70)
print("Searching for attention-related modules (keyword scan — verify, do")
print("not trust blindly; A3's lift search had a false positive on this")
print("exact kind of keyword match):")
print("=" * 70)
KEYWORDS = ["temporal", "attn", "attention", "rope", "position", "channel"]
for name, module in panda_baseline.named_modules():
    if any(k in name.lower() for k in KEYWORDS):
        print(f"  {name}  ->  {type(module).__name__}")

print()
print("=" * 70)
print("forward() signature of the top-level model:")
print("=" * 70)
try:
    print(inspect.signature(panda_baseline.forward))
except Exception as e:
    print("Could not introspect forward() directly:", e)
    print("Try panda_baseline.model.forward or similar based on the tree above.")


In [ ]:

# Cell 4b — shared constants, defined early since later cells (source
# inspection, hook smoke test) need CONTEXT_LEN before Part 3's data
# generators would otherwise define it.
CONTEXT_LEN = 512   # matches BASE_MODEL_CONFIG['context_length'] and Section 7's protocol
HORIZONS = [96, 192, 336]
N_WINDOWS = 20  # confirmed: matches A1/A3 confirmatory standard


In [ ]:

# Cell 6 — hook-based capture, confirmed working (round 1: all 8 layers
# captured real, non-None weights, shape (3, 8, 32, 32) on a 3-channel
# dummy input -- (batch*channels, heads, patches, patches)). Sidesteps
# trusting the full output_attentions threading chain by hooking
# temporal_self_attn directly at each of the 8 layers. Only
# temporal_self_attn is touched (channel_self_attn is out of scope --
# ruled out 4x already, Experiments 9/22/27/33).

import inspect

captured_temporal_attn = {}

def make_temporal_attn_hook(layer_idx):
    def hook(module, input, output):
        # output = (attn_output, attn_weights_reshaped, past_key_value)
        attn_weights = output[1]
        captured_temporal_attn[layer_idx] = (
            attn_weights.detach().cpu() if attn_weights is not None else None
        )
    return hook

def register_temporal_attn_hooks(model):
    handles = []
    for i, layer in enumerate(model.model.encoder.layers):
        h = layer.temporal_self_attn.register_forward_hook(make_temporal_attn_hook(i))
        handles.append(h)
    return handles

# Smoke test on baseline (already verified working -- rerun here for a
# fresh session's sanity check before trusting anything downstream).
hook_handles = register_temporal_attn_hooks(panda_baseline)
dummy_input = torch.randn(1, CONTEXT_LEN, 3, device=DEVICE)  # (batch, seq_len, channels)
captured_temporal_attn.clear()
with torch.no_grad():
    _ = panda_baseline(past_values=dummy_input, output_attentions=True)

all_ok = True
for i in range(len(panda_baseline.model.encoder.layers)):
    w = captured_temporal_attn.get(i)
    if w is None:
        print(f"Layer {i}: MISSING -- output_attentions did not reach this layer")
        all_ok = False
    else:
        print(f"Layer {i}: captured, shape {tuple(w.shape)}")

for h in hook_handles:
    h.remove()

assert all_ok, "STOP: not all layers captured attention weights."
print("\nAll layers captured successfully. Safe to proceed.")



## Part 3 — Data Generators (Reused Verbatim From Section 7 / A3 Where Possible)

Lorenz `gate_3ch` (fixed IC, manual RK4, 3 channels) and Burgers ν=1.0
(16-channel PCA, `load_burgers_nu1`) should be copy-pasted in verbatim from
your Section 7 / A1 100k-eval notebook rather than reimplemented here, to
guarantee this experiment is evaluated on exactly the same trajectories as
A1. Rossler, SprottB (seeded `solve_ivp`, matching A1's held-out-systems
protocol) and Harmonic likewise. Stub implementations below — **replace
with your actual Section 7 functions before running.**


In [ ]:

# Cell 7 — data generators (STUBS: replace with verbatim Section 7 / A1
# functions — do not reimplement from scratch, to avoid introducing a new
# trajectory that isn't the one A1's results are about)

def load_gate_3ch_lorenz():
    raise NotImplementedError("Paste verbatim from the Section 7 / A1 100k-eval notebook.")

def load_rossler():
    raise NotImplementedError("Paste verbatim from the Section 7 / A1 100k-eval notebook (held-out systems).")

def load_sprottb():
    raise NotImplementedError("Paste verbatim from the Section 7 / A1 100k-eval notebook (held-out systems).")

def load_burgers_nu1():
    raise NotImplementedError("Paste verbatim from Section 7's load_burgers_nu1 (16-channel PCA).")

def load_harmonic():
    raise NotImplementedError("Paste verbatim from the complexity-continuum / A3 harmonic generator.")

CLASS_LOADERS = {
    "Lorenz": load_gate_3ch_lorenz,
    "Rossler": load_rossler,
    "SprottB": load_sprottb,
    "Burgers_nu1": load_burgers_nu1,
    "Harmonic": load_harmonic,
}
# CONTEXT_LEN, HORIZONS, N_WINDOWS already defined earlier (Cell 4b)



## Part 4 — Probe 1: Patch-Order Shuffling

Permutes patch order in the context window and reassigns position indices
0..N-1 to match the *new* order (so p-RoPE encodes a coherent-but-false
sequence, not the trajectory's real absolute positions). Patch *content*
(and therefore the Koopman lift's output per patch) is untouched — this
isolates temporal attention's use of order from the lift itself, which A3
already tested separately.

Run on **both** checkpoints (baseline_100k, ablation_100k) per the
confirmed decision — this additionally tells us whether temporal
attention's order-sensitivity itself depends on the lift being present.


In [ ]:

# Cell 8 — patch shuffling utility
def shuffle_patches(context_window, patch_size=16, seed=None):
    \"\"\"context_window: (T, C) array. Returns a shuffled copy with patches
    permuted along the time axis, patch content preserved exactly.\"\"\"
    rng = np.random.default_rng(seed)
    T, C = context_window.shape
    assert T % patch_size == 0, f"context length {T} not divisible by patch_size {patch_size}"
    n_patches = T // patch_size
    patches = context_window.reshape(n_patches, patch_size, C)
    perm = rng.permutation(n_patches)
    shuffled = patches[perm].reshape(T, C)
    return shuffled, perm

def panda_forecast_with(pipe, context_np, horizon):
    # Verbatim from the user's own Section 7 / A1 harness -- do not modify.
    # NOTE: expects context_np CHANNEL-FIRST, shape (C, T) -- note the
    # internal ctx.T transpose below. This differs from the (T, C)
    # orientation used everywhere else in this notebook (shuffle_patches,
    # the data generators, Probe 3's direct model() calls), so the wrapper
    # below (panda_mae_forecast) is the single point where that conversion
    # happens -- do not call panda_forecast_with directly elsewhere in this
    # notebook without going through it, to avoid a silent transpose bug.
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = pipe.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)

def panda_mae_forecast(pipe, context_TC, horizon):
    \"\"\"Wrapper handling the (T,C) <-> (C,T) orientation conversion at a
    single explicit point. context_TC: (T, C), matches shuffle_patches /
    data-generator convention. Returns (horizon, C) to match target shape.\"\"\"
    context_CT = context_TC.T  # -> (C, T), what panda_forecast_with expects
    pred_CH = panda_forecast_with(pipe, context_CT, horizon)  # -> (C, horizon)
    return pred_CH.T  # -> (horizon, C)

def run_panda_forward_for_attention(model, context_window, horizon=96):
    \"\"\"Direct model() call used ONLY by Probe 3 to trigger the hooks
    registered in Cell 6/12 -- the forecast VALUE isn't used here (Probe 3
    only cares about the attention-weight side effect), so this
    deliberately does not attempt to parse the output object's prediction
    field at all, unlike an earlier version of this function.\"\"\"
    context_tensor = torch.tensor(
        context_window, dtype=torch.float32, device=DEVICE
    ).unsqueeze(0)  # (1, T, C)
    with torch.no_grad():
        _ = model(past_values=context_tensor, output_attentions=True)

def evaluate_shuffle_probe(class_name, loader_fn, pipe, model_name, horizon):
    trajectories = loader_fn()  # expects list of (T, C) arrays, len >= N_WINDOWS
    vanilla_maes, shuffled_maes = [], []
    for i in range(N_WINDOWS):
        context, target = trajectories[i]  # (CONTEXT_LEN, C), (horizon, C)
        vanilla_pred = panda_mae_forecast(pipe, context, horizon)
        shuffled_context, _ = shuffle_patches(context, seed=1000 + i)
        shuffled_pred = panda_mae_forecast(pipe, shuffled_context, horizon)
        vanilla_maes.append(np.mean(np.abs(vanilla_pred - target)))
        shuffled_maes.append(np.mean(np.abs(shuffled_pred - target)))
    vanilla_maes, shuffled_maes = np.array(vanilla_maes), np.array(shuffled_maes)
    degradation = shuffled_maes - vanilla_maes
    stat, p = wilcoxon(shuffled_maes, vanilla_maes, alternative="greater")
    return {
        "class": class_name, "model": model_name, "horizon": horizon,
        "vanilla_mae_median": np.median(vanilla_maes),
        "shuffled_mae_median": np.median(shuffled_maes),
        "degradation_median": np.median(degradation),
        "degradation_iqr": np.percentile(degradation, 75) - np.percentile(degradation, 25),
        "wilcoxon_p_shuffle_worse": p,
    }

shuffle_results = []
for class_name, loader_fn in CLASS_LOADERS.items():
    for pipe, model_name in [(pipe_baseline, "baseline_100k"), (pipe_ablation, "ablation_100k")]:
        for h in HORIZONS:
            shuffle_results.append(evaluate_shuffle_probe(class_name, loader_fn, pipe, model_name, h))

shuffle_df = pd.DataFrame(shuffle_results)
shuffle_df.to_csv("a2a_probe1_shuffle_results.csv", index=False)
shuffle_df



### Pre-registered gate check (Probe 1 only — run this exactly as specified, do not adjust after looking)


In [ ]:

# Cell 9 — pre-registered decision, baseline_100k only, per the criterion fixed above
gate_df = shuffle_df[(shuffle_df.model == "baseline_100k")]
chaotic_classes = ["Lorenz", "Rossler", "SprottB"]

verdict_rows = []
for h in HORIZONS:
    h_df = gate_df[gate_df.horizon == h]
    burgers_degr = h_df[h_df["class"] == "Burgers_nu1"]["degradation_median"].values[0]
    chaotic_degrs = h_df[h_df["class"].isin(chaotic_classes)]["degradation_median"].values
    chaotic_sig = h_df[h_df["class"].isin(chaotic_classes)]["wilcoxon_p_shuffle_worse"].values
    larger_and_sig = np.sum((chaotic_degrs > burgers_degr) & (chaotic_sig < 0.05))
    verdict_rows.append({
        "horizon": h, "burgers_degradation": burgers_degr,
        "chaotic_classes_larger_and_sig": f"{larger_and_sig}/3",
    })

verdict_df = pd.DataFrame(verdict_rows)
print(verdict_df)
n_horizons_supporting = sum(1 for r in verdict_rows if int(r["chaotic_classes_larger_and_sig"].split("/")[0]) >= 2)
print()
if n_horizons_supporting >= 2:
    print(f"PRE-REGISTERED VERDICT: ESCALATE TO A2b ({n_horizons_supporting}/3 horizons meet the criterion)")
else:
    print(f"PRE-REGISTERED VERDICT: DO NOT ESCALATE ({n_horizons_supporting}/3 horizons meet the criterion)")
print("This verdict is mechanical and was fixed before this notebook ran. "
      "Any override requires an explicit, separately-labeled addendum, not a "
      "silent edit of this cell's output.")



## Part 5 — Probe 2: Context-Length Truncation Curves

Dose-response: sweep context length, track **Panda's own** MAE degradation
per class (not a Panda-vs-Chronos advantage curve -- Chronos is
deliberately not loaded in this environment, see Cell 1). This is
exploratory/supporting evidence, not part of the pre-registered gate. If a
Panda-vs-Chronos advantage version is wanted later, run Chronos separately
against these same saved windows and join on CSV.


In [ ]:

# Cell 10 — truncation curve (Panda-only)
TRUNCATION_LENGTHS = [512, 256, 128, 64, 32]

def evaluate_truncation_probe(class_name, loader_fn, pipe, model_name, horizon):
    trajectories = loader_fn()
    rows = []
    for ctx_len in TRUNCATION_LENGTHS:
        panda_maes = []
        for i in range(N_WINDOWS):
            context, target = trajectories[i]
            truncated_context = context[-ctx_len:]  # (T,C), slices along time axis
            panda_pred = panda_mae_forecast(pipe, truncated_context, horizon)
            panda_maes.append(np.mean(np.abs(panda_pred - target)))
        rows.append({
            "class": class_name, "model": model_name, "horizon": horizon,
            "context_len": ctx_len, "panda_mae_median": np.median(panda_maes),
        })
    return rows

truncation_results = []
for class_name, loader_fn in CLASS_LOADERS.items():
    for pipe, model_name in [(pipe_baseline, "baseline_100k"), (pipe_ablation, "ablation_100k")]:
        for h in HORIZONS:
            truncation_results.extend(
                evaluate_truncation_probe(class_name, loader_fn, pipe, model_name, h)
            )

truncation_df = pd.DataFrame(truncation_results)
truncation_df.to_csv("a2a_probe2_truncation_results.csv", index=False)
truncation_df


In [ ]:

# Cell 11 — truncation curve plot, one panel per class, baseline vs ablation overlaid
fig, axes = plt.subplots(1, len(CLASS_LOADERS), figsize=(4 * len(CLASS_LOADERS), 4), sharey=False)
for ax, (class_name, _) in zip(axes, CLASS_LOADERS.items()):
    for model_name, style in [("baseline_100k", "-o"), ("ablation_100k", "--s")]:
        sub = truncation_df[(truncation_df["class"] == class_name) &
                             (truncation_df.model == model_name) &
                             (truncation_df.horizon == 96)]
        sub = sub.sort_values("context_len")
        ax.plot(sub.context_len, sub.panda_mae_median, style, label=model_name)
    ax.set_title(class_name)
    ax.set_xlabel("context length")
    ax.set_ylabel("Panda MAE (median)")
    ax.invert_xaxis()
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("a2a_probe2_truncation_curves.png", dpi=150)
plt.show()



## Part 6 — Probe 3: Attention-Map Inspection (Descriptive)

One representative window per class. Coded against the paper's own
vocabulary (Section 5.5): Toeplitz (shift-invariant/diagonal-banded),
block (attends within contiguous time ranges), selector (sharp attention
on a few key patches), hybrid. Summarised with attention entropy and an
effective-receptive-field proxy (fraction of total attention mass within
±k patches of the diagonal) rather than eyeballing heatmaps alone.


In [ ]:

# Cell 12 — attention extraction, all 8 layers, using the verified hook
# pattern from Cell 6 (register_temporal_attn_hooks / captured_temporal_attn).
# Reports per-layer summaries rather than collapsing to one number, since
# there is no a priori reason to expect all 8 layers to behave identically
# (earlier layers may attend more locally, later layers more globally --
# this is itself worth seeing, not just averaging away).

def get_all_layer_attention_maps(model, context_window):
    hook_handles = register_temporal_attn_hooks(model)
    captured_temporal_attn.clear()
    _ = run_panda_forward_for_attention(model, context_window, horizon=96)
    maps = {}
    for i in range(len(model.model.encoder.layers)):
        w = captured_temporal_attn.get(i)
        if w is None:
            raise RuntimeError(f"Layer {i} did not capture attention weights.")
        maps[i] = w.numpy()  # (batch*channels, heads, n_patches, n_patches)
    for h in hook_handles:
        h.remove()
    return maps

def attention_entropy(attn_map):
    \"\"\"attn_map: (..., n_patches, n_patches), last dim sums to 1 (post-softmax rows).\"\"\"
    eps = 1e-12
    ent = -(attn_map * np.log(attn_map + eps)).sum(axis=-1)
    return ent.mean()

def effective_receptive_field(attn_map, k=5):
    n = attn_map.shape[-1]
    band_mass = 0.0
    for i in range(n):
        lo, hi = max(0, i - k), min(n, i + k + 1)
        band_mass += attn_map[..., i, lo:hi].sum()
    return band_mass / attn_map.sum()

attn_summary_rows = []
for class_name, loader_fn in CLASS_LOADERS.items():
    trajectories = loader_fn()
    context, _ = trajectories[0]  # one representative window
    for model, model_name in [(panda_baseline, "baseline_100k"), (panda_ablation, "ablation_100k")]:
        layer_maps = get_all_layer_attention_maps(model, context)
        for layer_idx, attn_map in layer_maps.items():
            attn_summary_rows.append({
                "class": class_name, "model": model_name, "layer": layer_idx,
                "attn_entropy": attention_entropy(attn_map),
                "effective_receptive_field_frac": effective_receptive_field(attn_map),
            })

attn_summary_df = pd.DataFrame(attn_summary_rows)
attn_summary_df.to_csv("a2a_probe3_attention_summary.csv", index=False)
attn_summary_df


In [ ]:

# Cell 13 — visualize one attention heatmap per class, one representative
# layer (middle layer, index 3 of 0-7) for space; baseline checkpoint only.
fig, axes = plt.subplots(1, len(CLASS_LOADERS), figsize=(4 * len(CLASS_LOADERS), 4))
DISPLAY_LAYER = 3
for ax, (class_name, loader_fn) in zip(axes, CLASS_LOADERS.items()):
    trajectories = loader_fn()
    context, _ = trajectories[0]
    layer_maps = get_all_layer_attention_maps(panda_baseline, context)
    attn_map = layer_maps[DISPLAY_LAYER]
    # average over batch*channels and heads for display
    disp = attn_map.mean(axis=(0, 1))
    ax.imshow(disp, cmap="viridis")
    ax.set_title(f"{class_name} (layer {DISPLAY_LAYER})")
plt.tight_layout()
plt.savefig("a2a_probe3_attention_heatmaps.png", dpi=150)
plt.show()



## Part 7 — Summary and Next Steps

Fill in after running:
- Probe 1 pre-registered verdict (Cell 9 output): ESCALATE / DO NOT ESCALATE.
- Probe 2 (descriptive): does the truncation curve shape differ between
  chaotic-ODE classes and Burgers, and does it track the shuffle result?
- Probe 3 (descriptive): does attention structure (Toeplitz/block/selector,
  entropy, receptive field) differ systematically between classes where
  the lift helps vs. hurts, and between baseline vs. ablation checkpoints?

**Whatever the outcome, record it in the experiment log as Experiment 37+**
following the existing epistemic-labeling convention (OBS/PAT/HYP/SPEC/EST),
with the pre-registered verdict reported before any post-hoc interpretation,
per this project's established discipline.


In [ ]:

# Cell 14 — consolidated results export for the log addendum
all_results = {
    "probe1_shuffle": shuffle_df.to_dict(orient="records"),
    "probe1_gate_verdict": verdict_df.to_dict(orient="records"),
    "probe2_truncation": truncation_df.to_dict(orient="records"),
    "probe3_attention_summary": attn_summary_df.to_dict(orient="records"),
}
with open("a2a_full_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print("Saved a2a_full_results.json — attach this alongside the raw CSVs "
      "when writing up Experiment 37+ in the log.")
